In [1]:
import os
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

import pandas as pd
import json
import numpy as np
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn.functional as F

import os, sys
import gc
from torch.utils.data import DataLoader
from sklearn.metrics import (
    roc_auc_score, precision_recall_fscore_support, 
    accuracy_score, average_precision_score
)
from tqdm import tqdm
import pickle
import pandas as pd
import json as json_lib
from pathlib import Path
from dotenv import load_dotenv


In [2]:
from pathlib import Path
import zipfile

# Path to Kaggle input zip
zip_path = Path('/kaggle/input/notebooks/phmduyanh16/timeline-dataset/_output_.zip')

# Destination folder
extract_dir = Path('/kaggle/working/Timelines')

# Create destination
extract_dir.mkdir(parents=True, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("Extraction completed.")
print("Saved to:", extract_dir)


Extraction completed.
Saved to: /kaggle/working/Timelines


In [3]:
base_data_path = os.path.join('/kaggle/input/datasets/phmduyanh16/input-dataset')

N_DIAGNOSES = 200   # top-200 diagnosis 
N_DRUGS     = 50

pos_weight_mortality   = torch.tensor([63.93])   
pos_weight_los         = torch.tensor([4.39])    
pos_weight_readmission = torch.tensor([2.88])    
pos_weight_progression = torch.from_numpy(np.load(os.path.join(base_data_path, 'progression_pos_weights.npy'))).float()  # 200-dim

# Models
EMBED_DIM   = 128
HIDDEN_SIZE = 256
STATIC_DIM  = 64    # patient_vec and admission_vec each
PROJ_DIM    = 128
N_DIAGNOSES = 200

TIMELINE_DIR         = os.path.join('/kaggle/working/Timelines/data/Timelines')
ADMISSION_NODES_PATH = os.path.join(base_data_path, 'admission_nodes.json')
DIAG_VOCAB_PATH      = os.path.join(base_data_path, 'top200_diag_vocab.json')
PROG_WEIGHTS_PATH    = os.path.join(base_data_path, 'progression_pos_weights.npy')
TRAIN_DF_PATH        = os.path.join(base_data_path, 'train_df.csv')
VAL_DF_PATH          = os.path.join(base_data_path,  'val_df.csv')
TEST_DF_PATH         = os.path.join(base_data_path,  'test_df.csv')
PATIENT_CACHE_PATH   = os.path.join('/kaggle/input/datasets/phmduyanh16/caches/patient_cache.pt')
ADMISSION_CACHE_PATH = os.path.join(base_data_path,  'admission_cache.pt')
DRUG_WEIGHTS_PATH  = '/kaggle/input/datasets/phmduyanh16/drugss/drug_rec_pos_weights.npy'
DRUG_VOCAB_PATH = '/kaggle/input/datasets/phmduyanh16/drugss/top50_drug_vocab.json'

CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True, parents=True)

# Training hyperparameters
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE    = 64
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
PATIENCE      = 7        # early stopping patience
GRAD_CLIP     = 1.0      # max gradient norm
NUM_WORKERS   = 4
THRESHOLD     = 0.6
N_EPOCHS = 50


In [4]:
class EHRDataset(Dataset):
    """
    Args:
        admissions_df   : DataFrame with columns:
                          id, patient_id, inhospital_dead, los_log, los_7d,
                          readmission_30d
        timeline_dir    : path to Timelines/ folder
        admission_nodes : dict {adm_id (str): {'diagnoses': [...], 'drugs': [...]}}
        diag_to_idx     : dict {diagnosis_name (str, lower): int 0-199}
        drug_to_idx     : dict {drug_name (str, lower): int 0-49}
        patient_cache_tensor   : contiguous 2D Tensor (N, 64) — shared memory
        patient_to_idx         : dict {patient_id (str): row_idx}
        admission_cache_tensor : contiguous 2D Tensor (M, 64) — shared memory
        admission_to_idx       : dict {adm_id (str): row_idx}
    """

    def __init__(
        self,
        admissions_df,
        timeline_dir,
        admission_nodes,
        diag_to_idx,
        drug_to_idx,                  
        patient_cache_tensor,
        patient_to_idx,
        admission_cache_tensor,
        admission_to_idx,
        max_len=None,
        ablation_mode=None,
    ):
        self.timeline_dir           = Path(timeline_dir)
        self.admission_nodes        = admission_nodes
        self.diag_to_idx            = diag_to_idx
        self.drug_to_idx            = drug_to_idx   
        self.patient_cache_tensor   = patient_cache_tensor
        self.patient_to_idx         = patient_to_idx
        self.admission_cache_tensor = admission_cache_tensor
        self.admission_to_idx       = admission_to_idx
        self.max_len                = max_len
        self.ablation_mode          = ablation_mode

        # One row per admission — drop rows with missing critical labels
        df = admissions_df.copy()
        df = df[df['inhospital_dead'].notna()]
        df = df[df['los_log'].notna()]
        df['id']         = df['id'].astype(float).apply(lambda x: str(int(x)) if not pd.isna(x) else x)
        df['patient_id'] = df['patient_id'].astype(float).apply(lambda x: str(int(x)) if not pd.isna(x) else x)
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        import json  # Localized import to prevent NameError if globally imported differently
        
        row    = self.df.iloc[idx]
        adm_id = str(row['id'])
        pid    = str(row['patient_id'])

        # Load timeline metadata dynamically (no in-memory cache to prevent RAM leaks)
        meta_path = self.timeline_dir / f'{pid}_meta.json'
        if not meta_path.exists():
            return None
        with open(meta_path) as f:
            meta = json.load(f)

        discharge_pos = None
        for i, entry in enumerate(meta):
            if entry['type'] == 'DISCHARGE' and str(entry.get('adm_id')) == adm_id:
                discharge_pos = i
                break

        if discharge_pos is None:
            return None

        # Load and slice timeline causally up to DISCHARGE 
        emb_path = self.timeline_dir / f'{pid}_emb.npy'
        dt_path  = self.timeline_dir / f'{pid}_dt.npy'

        if not emb_path.exists() or not dt_path.exists():
            return None

        emb = np.load(emb_path)   # (T_full, 128) - Loaded directly to avoid file descriptor leaks
        dt  = np.load(dt_path)    # (T_full,)     - Loaded directly to avoid file descriptor leaks

        if np.isnan(emb).any():
            return None

        # Slice to include up to DISCHARGE (discharge_pos) and any trailing tokens of this admission
        meta_sliced = meta[:discharge_pos + 2]
        
        # Identify and remove the admission_emb of the current stay (leakage prevention)
        # We search backwards from the discharge token (discharge_pos - 1) to find the last admission_emb right before it
        remove_idx = None
        for i in range(discharge_pos - 1, -1, -1):
            entry = meta_sliced[i]
            if entry.get('type') == 'admission_emb' and str(entry.get('adm_id')) == adm_id:
                remove_idx = i
                break
                
        keep_indices = list(range(len(meta_sliced)))
        if remove_idx is not None:
            keep_indices.remove(remove_idx)
            
        emb = emb[keep_indices].copy()
        dt  = dt[keep_indices].copy()
        meta = [meta_sliced[idx] for idx in keep_indices]

        #─ ABLATION: Causal Slicing & Modality─
        if self.ablation_mode == 'last_24h':
            cum_time = np.cumsum(dt)
            total_time = cum_time[-1]
            mask = cum_time >= (total_time - 1.0)
            emb = emb[mask]
            dt  = dt[mask]
        elif self.ablation_mode == 'first_48h':
            cum_time = np.cumsum(dt)
            mask = cum_time <= 2.0
            emb = emb[mask]
            dt  = dt[mask]
        elif self.ablation_mode == 'static_only':
            emb = np.zeros_like(emb)
        elif self.ablation_mode == 'no_labs':
            emb = emb.copy()
            for i, entry in enumerate(meta[:len(emb)]):
                if entry.get('type', '').upper() == 'LAB':
                    emb[i] = 0
        elif self.ablation_mode == 'no_omr':
            emb = emb.copy()
            for i, entry in enumerate(meta[:len(emb)]):
                if entry.get('type', '').upper() == 'OMR':
                    emb[i] = 0
        elif self.ablation_mode == 'no_outnotes':
            emb = emb.copy()
            for i, entry in enumerate(meta[:len(emb)]):
                if entry.get('type', '').upper() == 'OUTNOTE':
                    emb[i] = 0
        elif self.ablation_mode == 'no_icu':
            emb = emb.copy()
            for i, entry in enumerate(meta[:len(emb)]):
                if entry.get('type', '').upper() == 'ICU':
                    emb[i] = 0
        elif self.ablation_mode == 'no_transfers':
            emb = emb.copy()
            for i, entry in enumerate(meta[:len(emb)]):
                if entry.get('type', '').upper() == 'TRANSFER':
                    emb[i] = 0
        elif self.ablation_mode == 'no_last_event':
            if len(emb) > 0:
                emb = emb[:-1]
                dt  = dt[:-1]
        elif self.ablation_mode == 'no_future':
            if len(emb) >= 2:
                emb = emb[:-2]
                dt  = dt[:-2]
            elif len(emb) > 0:
                emb = emb[:-1]
                dt  = dt[:-1]

        # Causal capping
        if self.max_len is not None and len(emb) > self.max_len:
            emb = emb[-self.max_len:]
            dt  = dt[-self.max_len:]

        emb = emb.copy()
        dt  = dt.copy()

        # Static vectors from precomputed cache 
        if self.ablation_mode == 'no_static' or self.ablation_mode == 'static_only':
            patient_vec = torch.zeros(64)
            admission_vec = torch.zeros(64)
        elif self.ablation_mode == 'no_patient':
            patient_vec = torch.zeros(64)
            a_idx = self.admission_to_idx.get(str(adm_id))
            if a_idx is not None:
                admission_vec = self.admission_cache_tensor[a_idx]
            else:
                admission_vec = torch.zeros(64)
        elif self.ablation_mode == 'no_admission' or self.ablation_mode == 'no_future':
            p_idx = self.patient_to_idx.get(str(pid))
            if p_idx is not None:
                patient_vec = self.patient_cache_tensor[p_idx]
            else:
                patient_vec = torch.zeros(64)
            admission_vec = torch.zeros(64)
        else:
            p_idx = self.patient_to_idx.get(str(pid))
            if p_idx is not None:
                patient_vec = self.patient_cache_tensor[p_idx]
            else:
                patient_vec = torch.zeros(64)

            a_idx = self.admission_to_idx.get(str(adm_id))
            if a_idx is not None:
                admission_vec = self.admission_cache_tensor[a_idx]
            else:
                admission_vec = torch.zeros(64)

        if patient_vec is None or admission_vec is None:
            return None

        if torch.isnan(patient_vec).any() or torch.isnan(admission_vec).any():
            return None

        # Build progression multilabel vector (200,)
        progression = np.zeros(N_DIAGNOSES, dtype=np.float32)
        adm_data    = self.admission_nodes.get(adm_id, {})
        for diag in adm_data.get('diagnoses', []):
            i = self.diag_to_idx.get(diag.lower())
            if i is not None:
                progression[i] = 1.0

        # Build drug_rec multilabel vector (50,)
        drug_rec = np.zeros(N_DRUGS, dtype=np.float32)
        for drug in adm_data.get('drugs', []):
            i = self.drug_to_idx.get(drug.lower())
            if i is not None:
                drug_rec[i] = 1.0

        mortality   = float(row['inhospital_dead'])
        los_log     = float(row['los_log'])
        los_7d      = float(row['los_7d'])
        readmission = float(row['readmission_30d']) if not np.isnan(row['readmission_30d']) else -1.0

        return {
            'emb':           torch.tensor(emb,          dtype=torch.float32),
            'dt':            torch.tensor(dt,            dtype=torch.float32),
            'patient_vec':   patient_vec,
            'admission_vec': admission_vec,
            'mortality':     torch.tensor(mortality,     dtype=torch.float32),
            'los_log':       torch.tensor(los_log,       dtype=torch.float32),
            'los_7d':        torch.tensor(los_7d,        dtype=torch.float32),
            'readmission':   torch.tensor(readmission,   dtype=torch.float32),
            'progression':   torch.tensor(progression,   dtype=torch.float32),
            'drug_rec':      torch.tensor(drug_rec,      dtype=torch.float32),
            'adm_id':        adm_id,
            'pid':           pid,
            'ablation_mode': self.ablation_mode,
        }


In [5]:
class TimeEncoding(nn.Module):
    """
    Learns a continuous embedding for time deltas (dt).
    Uses a combination of linear and periodic components.
    """
    def __init__(self, d_model):
        super().__init__()
        self.linear = nn.Linear(1, 1)
        self.periodic = nn.Linear(1, d_model - 1)
        
    def forward(self, dt):
        # dt shape: (B, T)
        dt = dt.unsqueeze(-1) # (B, T, 1)
        
        v1 = self.linear(dt) # Linear component
        v2 = torch.sin(self.periodic(dt)) # Periodic components
        
        return torch.cat([v1, v2], dim=-1) # (B, T, d_model)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


In [6]:
class EHRTransformer(nn.Module):
    """
    Upgraded Transformer-based sequence modeling for EHR data.
    Injects static vectors as tokens and encodes relative time deltas (dt).
    """
    def __init__(
        self,
        n_diagnoses: int = N_DIAGNOSES,
        n_drugs: int     = N_DRUGS,
        dropout: float   = 0.1,
        lambda_init: float = 0.1,
        target_task: str = 'all'
    ):
        super().__init__()
        self.target_task = target_task

        self.log_lambda = nn.Parameter(torch.tensor(lambda_init).log())

        # Projections
        self.input_proj = nn.Linear(EMBED_DIM, HIDDEN_SIZE)
        self.static_proj = nn.Linear(STATIC_DIM, HIDDEN_SIZE) 
        
        # LSTM replaces Sinusoidal Positional Encoding
        self.pos_lstm = nn.LSTM(
            input_size=HIDDEN_SIZE,
            hidden_size=HIDDEN_SIZE,
            num_layers=2,
            batch_first=True,
            dropout=dropout if dropout > 0 else 0
        )
        
        # Transformer layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=HIDDEN_SIZE,
            nhead=8,
            dim_feedforward=HIDDEN_SIZE * 4,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)

        self.use_gradient_checkpointing = False

        self.proj = nn.Sequential(
            nn.Linear(HIDDEN_SIZE, PROJ_DIM),
            nn.LayerNorm(PROJ_DIM),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        if target_task in ['all', 'mortality']:
            self.head_mortality   = nn.Linear(PROJ_DIM, 1)
        if target_task in ['all', 'mortality']:
            self.head_mortality   = nn.Linear(PROJ_DIM, 1)
        if target_task in ['all', 'los_7d']:
            self.head_los         = nn.Linear(PROJ_DIM, 1)
        if target_task in ['all', 'readmission']:
            self.head_readmission = nn.Linear(PROJ_DIM, 1)
        if target_task in ['all', 'progression']:
            self.head_progression = nn.Linear(PROJ_DIM, n_diagnoses)
        if target_task in ['all', 'drug_rec']:
            self.head_drug_rec    = nn.Linear(PROJ_DIM + n_diagnoses, n_drugs)

        # Static task-specific pooling weights (alphas)
        self.alpha_mortality   = nn.Parameter(torch.tensor(0.0))
        self.alpha_los         = nn.Parameter(torch.tensor(0.0))
        self.alpha_readm       = nn.Parameter(torch.tensor(0.0))
        self.alpha_prog        = nn.Parameter(torch.tensor(0.0))
        self.alpha_drug        = nn.Parameter(torch.tensor(0.0))

    def forward(self, batch):
        emb           = batch['emb']            # (B, T, 128)
        dt            = batch['dt']             # (B, T)
        lengths       = batch['lengths']        # (B,)
        patient_vec   = batch['patient_vec']    # (B, 64)
        admission_vec = batch['admission_vec']  # (B, 64)

        # Temporal Decay (Learnable exponential decay)
        if batch.get('ablation_mode') == 'no_dt_decay':
            decay = torch.ones_like(dt).unsqueeze(-1)
        else:
            lam = torch.nn.functional.softplus(self.log_lambda)
            decay = torch.exp(-lam * dt).unsqueeze(-1)
        emb   = emb * decay

        # 2. Project clinical embeddings
        x = self.input_proj(emb)                # (B, T, 256)
        
        # 3. LSTM-based Sequential Encoding (Replaces Positional/Time Encoding)
        # We pack the sequence to ignore padding during the LSTM pass
        packed_x = torch.nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        lstm_out_packed, _ = self.pos_lstm(packed_x)
        x, _ = torch.nn.utils.rnn.pad_packed_sequence(
            lstm_out_packed, batch_first=True, total_length=emb.shape[1]
        )                                       # (B, T, 256)

        # 4. Project Static Contexts
        p_tok = self.static_proj(patient_vec).unsqueeze(1)    # (B, 1, 256)
        a_tok = self.static_proj(admission_vec).unsqueeze(1)  # (B, 1, 256)

        # 5. Prepend Static Context Tokens
        x_full = torch.cat([p_tok, a_tok, x], dim=1) # (B, T+2, 256)
        
        # Masking
        B, T_full, _ = x_full.shape
        # Adjust mask to account for 2 prepended tokens (which are never masked)
        mask = torch.zeros((B, T_full), dtype=torch.bool, device=x.device)
        for i, length in enumerate(lengths):
            mask[i, length+2:] = True
        
        # Transformer or MLP Pass
        if batch.get('ablation_mode') == 'no_temporal':
            # BAG OF EVENTS ABLATION: Ignore order/attention, just mean pool
            # x_full: (B, T+2, HIDDEN_SIZE)
            trans_out = x_full 
        else:
            trans_out = self.transformer(x_full, src_key_padding_mask=mask)

        # Hybrid Global Representation
        # Use the Patient_Token (index 0) as it has now attended to everything
        # Plus the last clinical token (discharge)
        idx_discharge = (lengths + 1).clamp(min=1) # +1 because of 2 prepended tokens
        idx_expanded = idx_discharge.view(-1, 1, 1).expand(-1, 1, HIDDEN_SIZE)
        h_discharge = trans_out.gather(1, idx_expanded).squeeze(1)
        
        h_global = trans_out[:, 0] # The evolved Patient Token
        
        out = {}
        
        # Helper for task-specific pooling with static alphas
        def get_task_repr(alpha_param):
            gate_val = torch.sigmoid(alpha_param)
            pooled = gate_val * h_global + (1.0 - gate_val) * h_discharge
            return self.proj(pooled)

        # 1. Mortality
        if self.target_task in ['all', 'mortality']:
            s_mort = get_task_repr(self.alpha_mortality)
            out['mortality'] = self.head_mortality(s_mort)
        
        # 2. LOS
        if self.target_task in ['all', 'los_7d']:
            s_los = get_task_repr(self.alpha_los)
            out['los_7d'] = self.head_los(s_los)
            
        # 3. Readmission
        if self.target_task in ['all', 'readmission']:
            s_readm = get_task_repr(self.alpha_readm)
            out['readmission'] = self.head_readmission(s_readm)
            
        # 4. Progression (Diagnoses)
        prog_logits = None
        if self.target_task in ['all', 'progression', 'drug_rec']:
            s_prog = get_task_repr(self.alpha_prog)
            prog_logits = self.head_progression(s_prog)
            if self.target_task in ['all', 'progression']:
                out['progression'] = prog_logits
        
        # 5. Drug Recommendation (Dependent on Progression)
        if self.target_task in ['all', 'drug_rec']:
            s_drug = get_task_repr(self.alpha_drug)
            combined_drug = torch.cat([s_drug, prog_logits], dim=-1)
            out['drug_rec'] = self.head_drug_rec(combined_drug)
        
        out['shared_repr'] = (h_global + h_discharge) / 2.0 # For backward compatibility in logs
        return out


In [7]:
class EHRTransformerBase(nn.Module):
    """
    Original Transformer-based sequence modeling for EHR data.
    Uses late fusion: concatenates static vectors AFTER the transformer pass.
    Matches the state_dict in checkpoints/transformer_base/best_model.pt
    """
    def __init__(
        self,
        n_diagnoses: int = N_DIAGNOSES,
        n_drugs: int     = N_DRUGS,
        dropout: float   = 0.1,
        lambda_init: float = 0.1,
        target_task: str = 'all'
    ):
        super().__init__()
        self.target_task = target_task

        # Δt decay parameter — learned scalar
        self.log_lambda = nn.Parameter(torch.tensor(lambda_init).log())

        # Projections
        self.input_proj = nn.Linear(EMBED_DIM, HIDDEN_SIZE)
        
        self.pos_encoder = PositionalEncoding(HIDDEN_SIZE)
        
        # Transformer layers (3 layers based on checkpoint)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=HIDDEN_SIZE,
            nhead=8,
            dim_feedforward=HIDDEN_SIZE * 4,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)

        # Projection: concat(transformer_out, patient, admission) -> shared repr
        # 256 (Hidden) + 64 (Patient) + 64 (Admission) = 384
        concat_dim = HIDDEN_SIZE + STATIC_DIM + STATIC_DIM 
        self.proj = nn.Sequential(
            nn.Linear(concat_dim, PROJ_DIM),
            nn.LayerNorm(PROJ_DIM)
        )

        if target_task in ['all', 'mortality']:
            self.head_mortality   = nn.Linear(PROJ_DIM, 1)
        if target_task in ['all', 'los_7d']:
            self.head_los         = nn.Linear(PROJ_DIM, 1)
        if target_task in ['all', 'readmission']:
            self.head_readmission = nn.Linear(PROJ_DIM, 1)
        if target_task in ['all', 'progression']:
            self.head_progression = nn.Linear(PROJ_DIM, n_diagnoses)
        if target_task in ['all', 'drug_rec']:
            self.head_drug_rec    = nn.Linear(PROJ_DIM, n_drugs)

    def forward(self, batch):
        emb           = batch['emb']            # (B, T, 128)
        dt            = batch['dt']             # (B, T)
        lengths       = batch['lengths']        # (B,)
        patient_vec   = batch['patient_vec']    # (B, 64)
        admission_vec = batch['admission_vec']  # (B, 64)

        # Temporal Decay
        lam = torch.nn.functional.softplus(self.log_lambda)
        decay = torch.exp(-lam * dt).unsqueeze(-1)
        emb   = emb * decay

        # Project + Positional Encoding
        x = self.input_proj(emb)                # (B, T, 256)
        x = self.pos_encoder(x)
        
        # Masking
        B, T, _ = x.shape
        mask = torch.zeros((B, T), dtype=torch.bool, device=x.device)
        for i, length in enumerate(lengths):
            mask[i, length:] = True
        
        # Transformer Pass
        trans_out = self.transformer(x, src_key_padding_mask=mask)

        # Global Representation: Slice at the last real token (DISCHARGE position)
        idx          = (lengths - 1).clamp(min=0)
        idx_expanded = idx.view(-1, 1, 1).expand(-1, 1, HIDDEN_SIZE)
        h_discharge  = trans_out.gather(1, idx_expanded).squeeze(1)  # (B, 256)

        # Late Fusion: Concat static vectors + project
        combined = torch.cat([h_discharge, patient_vec, admission_vec], dim=-1) # (B, 384)
        shared = self.proj(combined)   # (B, 128)

        out = {}
        if self.target_task in ['all', 'mortality']:
            out['mortality'] = self.head_mortality(shared)
        if self.target_task in ['all', 'los_7d']:
            out['los_7d'] = self.head_los(shared)
        if self.target_task in ['all', 'readmission']:
            out['readmission'] = self.head_readmission(shared)
        if self.target_task in ['all', 'progression']:
            out['progression'] = self.head_progression(shared)
        if self.target_task in ['all', 'drug_rec']:
            out['drug_rec'] = self.head_drug_rec(shared)
        
        out["shared_repr"] = h_discharge
        return out


In [8]:
class EHRModel(nn.Module):
    """
    Args:
        n_diagnoses (int) : size of progression label space (default 200)
        n_drugs (int)     : size of drug recommendation label space (default 50)
        dropout (float)   : dropout rate in projection and heads
        lambda_init (float): initial value for Δt decay parameter λ
                             learned during training
    """

    def __init__(
        self,
        n_diagnoses: int   = N_DIAGNOSES,
        n_drugs: int       = N_DRUGS,       # ← NEW
        dropout: float     = 0.1,
        lambda_init: float = 0.1,
        target_task: str   = 'all'
    ):
        super().__init__()
        self.target_task = target_task

        # Δt decay parameter — learned scalar
        # emb_t = emb_t * exp(-λ * Δt)
        # λ > 0 enforced via softplus in forward
        self.log_lambda = nn.Parameter(torch.tensor(lambda_init).log())

        # lstm
        self.lstm = nn.LSTM(
            input_size  = EMBED_DIM,    # 128
            hidden_size = HIDDEN_SIZE,  # 256
            num_layers  = 2,
            batch_first = True,
            dropout     = 0.5,          # dropout handled outside for single layer
        )

        self.use_gradient_checkpointing = False

        # Projection: concat → shared repr
        concat_dim = HIDDEN_SIZE + STATIC_DIM + STATIC_DIM  # 256 + 64 (patient) + 64 (admission) = 384
        self.proj = nn.Sequential(
            nn.Linear(concat_dim, PROJ_DIM),
            nn.LayerNorm(PROJ_DIM),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # Prediction heads — all output raw logits, sigmoid/BCE applied in loss
        # Prediction heads
        if target_task in ['all', 'mortality']:
            self.head_mortality   = nn.Linear(PROJ_DIM, 1)
        if target_task in ['all', 'los_7d']:
            self.head_los         = nn.Linear(PROJ_DIM, 1)
        if target_task in ['all', 'readmission']:
            self.head_readmission = nn.Linear(PROJ_DIM, 1)
        if target_task in ['all', 'progression']:
            self.head_progression = nn.Linear(PROJ_DIM, n_diagnoses)
        if target_task in ['all', 'drug_rec']:
            self.head_drug_rec    = nn.Linear(PROJ_DIM, n_drugs)

    def forward(self, batch):
        """
        Args:
            batch (dict) from ehr_collate_fn:
                emb           : (B, T, 128)  padded timeline embeddings
                dt            : (B, T)       Δt values in days
                lengths       : (B,)         true sequence lengths
                patient_vec   : (B, 64)
                admission_vec : (B, 64)

        Returns:
            dict of logits:
                mortality   : (B, 1)
                los_7d      : (B, 1)
                readmission : (B, 1)
                progression : (B, 200)
                drug_rec    : (B, 50)
        """
        emb           = batch['emb']            # (B, T, 128)
        dt            = batch['dt']             # (B, T)
        lengths       = batch['lengths']        # (B,)
        patient_vec   = batch['patient_vec']    # (B, 64)
        admission_vec = batch['admission_vec']  # (B, 64)

        # Apply exponential Δt decay
        # λ = softplus(log_lambda) ensures λ > 0
        lam = torch.nn.functional.softplus(self.log_lambda)

        # decay shape: (B, T, 1) → broadcast over 128 dims
        decay = torch.exp(-lam * dt).unsqueeze(-1)   # (B, T, 1)
        emb   = emb * decay                           # (B, T, 128)

        # lstm with packed sequence (ignores padding)
        packed = pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        # lstm pass
        if self.use_gradient_checkpointing and self.training:
            from torch.utils.checkpoint import checkpoint
            def lstm_forward(x):
                out, _ = self.lstm(x)
                return out
            lstm_out_packed = checkpoint(lstm_forward, packed, use_reentrant=False)
        else:
            lstm_out_packed, _ = self.lstm(packed)

        lstm_out, _ = nn.utils.rnn.pad_packed_sequence(
            lstm_out_packed, batch_first=True
        )                                            # (B, T, 256)

        # Slice hidden state at last real token (DISCHARGE position)
        # lengths[i] - 1 is the index of the DISCHARGE token since we slice
        # the timeline up to and including DISCHARGE in EHRDataset
        idx          = (lengths - 1).clamp(min=0)             # (B,)
        idx_expanded = idx.view(-1, 1, 1).expand(-1, 1, HIDDEN_SIZE)
        h_discharge  = lstm_out.gather(1, idx_expanded).squeeze(1)  # (B, 256)

        # Concat static vectors + project
        combined = torch.cat([h_discharge, patient_vec, admission_vec], dim=-1)
        # (B, 384)
        shared = self.proj(combined)   # (B, 128)

        # DEBUG: Check for NaNs
        if torch.isnan(shared).any():
            print(f"\n[DEBUG] NaN detected in model output!")
            print(f"  - emb max/min: {emb.max().item():.2f}/{emb.min().item():.2f}")
            print(f"  - dt max/min: {dt.max().item():.2f}/{dt.min().item():.2f}")
            print(f"  - lam: {lam.item():.4f}")
            print(f"  - h_discharge max/min: {h_discharge.max().item():.2f}/{h_discharge.min().item():.2f}")
            print(f"  - patient_vec max/min: {patient_vec.max().item():.2f}/{patient_vec.min().item():.2f}")
            print(f"  - admission_vec max/min: {admission_vec.max().item():.2f}/{admission_vec.min().item():.2f}")

        # 5 prediction heads
        out = {}
        if self.target_task in ['all', 'mortality']:
            out['mortality'] = self.head_mortality(shared)
        if self.target_task in ['all', 'los_7d']:
            out['los_7d'] = self.head_los(shared)
        if self.target_task in ['all', 'readmission']:
            out['readmission'] = self.head_readmission(shared)
        if self.target_task in ['all', 'progression']:
            out['progression'] = self.head_progression(shared)
        if self.target_task in ['all', 'drug_rec']:
            out['drug_rec'] = self.head_drug_rec(shared)
        
        out["shared_repr"] = h_discharge
        return out


In [9]:
class ClinicalGAT(nn.Module):
    """
    Message Passing layer for EHR Graphs.
    Aggregates Labs, OMR, and Admission embeddings into a unified state.
    """
    def __init__(self, feature_dim=128, heads=4):
        super().__init__()
        self.feature_dim = feature_dim
        self.heads = heads
        self.head_dim = feature_dim // heads
        
        self.q = nn.Linear(feature_dim, feature_dim)
        self.k = nn.Linear(feature_dim, feature_dim)
        self.v = nn.Linear(feature_dim, feature_dim)
        
        self.out_proj = nn.Linear(feature_dim, feature_dim)
        self.ln = nn.LayerNorm(feature_dim)
        
    def forward(self, nodes):
        # nodes: (Batch, N_nodes, feature_dim)
        # Simplified GAT using Scaled Dot-Product Attention
        b, n, c = nodes.shape
        
        q = self.q(nodes).view(b, n, self.heads, self.head_dim).transpose(1, 2)
        k = self.k(nodes).view(b, n, self.heads, self.head_dim).transpose(1, 2)
        v = self.v(nodes).view(b, n, self.heads, self.head_dim).transpose(1, 2)
        
        # Attention scores (B, Heads, N, N)
        attn = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        
        out = (attn @ v).transpose(1, 2).reshape(b, n, c)
        out = self.ln(nodes + self.out_proj(out))
        return out

import torch.nn.functional as F

class BinaryFocalLoss(nn.Module):
    """
    Binary Focal Loss for highly imbalanced clinical classification tasks.
    FL(p_t) = -alpha * (1 - p_t)^gamma * log(p_t)
    """
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        inputs = inputs.float()
        targets = targets.float()
        
        # Calculate standard BCE loss without reduction
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        
        p = torch.sigmoid(inputs)
        p_t = p * targets + (1 - p) * (1 - targets) # Probability of correct class
        
        # Apply the focal modulating factor
        loss = bce_loss * ((1 - p_t) ** self.gamma)
        
        if self.alpha >= 0:
            alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
            loss = alpha_t * loss
            
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

# Loss function
class EHRLoss(nn.Module):
    """
    Multi-task BCE loss with per-task pos_weight and label masking.

    Masking rules:
        readmission : label == -1.0 → excluded (last admission / patient died)
        progression : all-zero vector → excluded (no top-200 diagnoses)
        drug_rec    : all-zero vector → excluded (no top-50 drugs)

    Task weights allow tuning relative importance during training.
    Default: all tasks weighted equally.
    """

    def __init__(
        self,
        pos_weight_mortality:   torch.Tensor,   # (1,)
        pos_weight_los:         torch.Tensor,   # (1,)
        pos_weight_readmission: torch.Tensor,   # (1,)
        pos_weight_progression: torch.Tensor,   # (200,)
        pos_weight_drug_rec:    torch.Tensor,   # (50,)   ← NEW
        w_mortality:   float = 1.0,
        w_los:         float = 1.0,
        w_readmission: float = 1.0,
        w_progression: float = 1.0,
        w_drug_rec:    float = 1.0,             # ← NEW
        use_focal_loss_mortality: bool = False,
    ):
        super().__init__()

        self.w_mortality   = w_mortality
        self.w_los         = w_los
        self.w_readmission = w_readmission
        self.w_progression = w_progression
        self.w_drug_rec    = w_drug_rec         # ← NEW

        # Scalar tasks — standard weighted BCE or Focal Loss
        if use_focal_loss_mortality:
            self.crit_mortality = BinaryFocalLoss(alpha=0.25, gamma=2.0)
        else:
            self.crit_mortality = nn.BCEWithLogitsLoss(pos_weight=pos_weight_mortality)
            
        self.crit_los         = nn.BCEWithLogitsLoss(pos_weight=pos_weight_los)
        self.crit_readmission = nn.BCEWithLogitsLoss(pos_weight=pos_weight_readmission)

        # Multilabel tasks — reduction='none' so we can mask per-sample
        # pos_weight broadcasts over the label dimension automatically
        self.crit_progression = nn.BCEWithLogitsLoss(
            pos_weight=pos_weight_progression, reduction='none'
        )
        self.crit_drug_rec = nn.BCEWithLogitsLoss(   # ← NEW
            pos_weight=pos_weight_drug_rec, reduction='none'
        )

    def _masked_multilabel_loss(self, criterion, logits, labels):
        """
        Compute mean multilabel BCE loss over samples that have at least
        one positive label. Samples with all-zero labels are excluded.

        Args:
            criterion : BCEWithLogitsLoss(reduction='none')
            logits    : (B, C)
            labels    : (B, C)

        Returns:
            scalar loss or 0.0 if no valid samples
        """
        mask = (labels.sum(dim=-1) > 0)   # (B,) — True if sample has ≥1 positive
        if mask.any():
            loss_unreduced = criterion(logits[mask], labels[mask])  # (n_valid, C)
            return loss_unreduced.mean()
        return torch.tensor(0.0, device=logits.device)

    def forward(self, logits: dict, batch: dict):
        """
        Returns:
            total_loss : scalar
            loss_dict  : {task: scalar} for logging
        """
        total = 0.0
        loss_dict = {'total': 0.0}

        # Mortality
        if 'mortality' in logits:
            labels_mort = batch['mortality'].unsqueeze(1)
            loss_mort = self.crit_mortality(logits['mortality'], labels_mort)
            total += self.w_mortality * loss_mort
            loss_dict['mortality'] = loss_mort.item()
        else:
            loss_dict['mortality'] = 0.0

        # LOS
        if 'los_7d' in logits:
            labels_los = batch['los_7d'].unsqueeze(1)
            loss_los = self.crit_los(logits['los_7d'], labels_los)
            total += self.w_los * loss_los
            loss_dict['los_7d'] = loss_los.item()
        else:
            loss_dict['los_7d'] = 0.0

        # Readmission
        if 'readmission' in logits:
            labels_readm = batch['readmission'].unsqueeze(1)
            readm_mask = (labels_readm >= 0)
            if readm_mask.any():
                loss_readm = self.crit_readmission(logits['readmission'][readm_mask], labels_readm[readm_mask])
            else:
                loss_readm = torch.tensor(0.0, device=logits['readmission'].device)
            total += self.w_readmission * loss_readm
            loss_dict['readmission'] = loss_readm.item()
        else:
            loss_dict['readmission'] = 0.0

        # Progression
        if 'progression' in logits:
            labels_prog = batch['progression']
            loss_prog = self._masked_multilabel_loss(self.crit_progression, logits['progression'], labels_prog)
            total += self.w_progression * loss_prog
            loss_dict['progression'] = loss_prog.item()
        else:
            loss_dict['progression'] = 0.0

        # Drug rec
        if 'drug_rec' in logits:
            labels_drug = batch['drug_rec']
            loss_drug = self._masked_multilabel_loss(self.crit_drug_rec, logits['drug_rec'], labels_drug)
            total += self.w_drug_rec * loss_drug
            loss_dict['drug_rec'] = loss_drug.item()
        else:
            loss_dict['drug_rec'] = 0.0

        loss_dict['total'] = total.item() if isinstance(total, torch.Tensor) else total
        return total, loss_dict



In [10]:
def ehr_collate_fn(batch):
    """
    Pads variable-length timelines to max T in batch.
    Filters out None samples.
    """
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None

    max_T = max(b['emb'].shape[0] for b in batch)

    emb_padded, dt_padded, lengths = [], [], []
    for b in batch:
        T   = b['emb'].shape[0]
        pad = max_T - T
        emb_padded.append(torch.nn.functional.pad(b['emb'], (0, 0, 0, pad)))
        dt_padded.append(torch.nn.functional.pad(b['dt'],   (0, pad)))
        lengths.append(T)

    return {
        'emb':           torch.stack(emb_padded),                               # (B, max_T, 128)
        'dt':            torch.stack(dt_padded),                                # (B, max_T)
        'lengths':       torch.tensor(lengths, dtype=torch.long),               # (B,)
        'patient_vec':   torch.stack([b['patient_vec']   for b in batch]),      # (B, 64)
        'admission_vec': torch.stack([b['admission_vec'] for b in batch]),      # (B, 64)
        'mortality':     torch.stack([b['mortality']     for b in batch]),      # (B,)
        'los_log':       torch.stack([b['los_log']       for b in batch]),      # (B,)
        'los_7d':        torch.stack([b['los_7d']        for b in batch]),      # (B,)
        'readmission':   torch.stack([b['readmission']   for b in batch]),      # (B,)
        'progression':   torch.stack([b['progression']   for b in batch]),      # (B, 200)
        'drug_rec':      torch.stack([b['drug_rec']      for b in batch]),      # (B, 50)
        'adm_ids':       [b['adm_id'] for b in batch],
        'pids':          [b['pid']    for b in batch],
        'ablation_mode': batch[0].get('ablation_mode'),
    }


In [11]:
def evaluate(model, loader, criterion, device):
    model.eval()

    tasks      = ['mortality', 'los_7d', 'readmission', 'progression', 'drug_rec']
    all_logits = {t: [] for t in tasks}
    all_labels = {t: [] for t in tasks}
    total_loss = {t: 0.0 for t in ['total', 'mortality', 'los_7d', 'readmission', 'progression', 'drug_rec']}
    n_batches  = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating', leave=False):
            if batch is None: continue
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            
            with torch.amp.autocast('cuda'):
                logits = model(batch)
                loss, loss_dict = criterion(logits, batch)
                
            for k, v in loss_dict.items(): total_loss[k] += v
            n_batches += 1

            # Mortality
            if 'mortality' in logits:
                all_logits['mortality'].append(torch.sigmoid(logits['mortality']).squeeze(1).cpu().numpy())
                all_labels['mortality'].append(batch['mortality'].cpu().numpy())

            # LOS
            if 'los_7d' in logits:
                all_logits['los_7d'].append(torch.sigmoid(logits['los_7d']).squeeze(1).cpu().numpy())
                all_labels['los_7d'].append(batch['los_7d'].cpu().numpy())

            # Readmission (Mask missing -1 labels)
            if 'readmission' in logits:
                readm_mask = batch['readmission'] >= 0
                if readm_mask.any():
                    all_logits['readmission'].append(torch.sigmoid(logits['readmission']).squeeze(1)[readm_mask].cpu().numpy())
                    all_labels['readmission'].append(batch['readmission'][readm_mask].cpu().numpy())

            # Progression (Mask empty samples)
            if 'progression' in logits:
                prog_mask = batch['progression'].sum(dim=-1) > 0
                if prog_mask.any():
                    all_logits['progression'].append(torch.sigmoid(logits['progression'])[prog_mask].cpu().numpy())
                    all_labels['progression'].append(batch['progression'][prog_mask].cpu().numpy())

            # Drug rec (Mask empty samples)
            if 'drug_rec' in logits:
                drug_mask = batch['drug_rec'].sum(dim=-1) > 0
                if drug_mask.any():
                    all_logits['drug_rec'].append(torch.sigmoid(logits['drug_rec'])[drug_mask].cpu().numpy())
                    all_labels['drug_rec'].append(batch['drug_rec'][drug_mask].cpu().numpy())

    metrics = {}

    # Binary tasks (Mortality, LOS, Readmission)
    for task in ['mortality', 'los_7d', 'readmission']:
        if len(all_labels[task]) == 0:
            metrics[task] = metrics[f'{task}_aupr'] = metrics[f'{task}_mAP'] = 0.0
            metrics[f'{task}_f1'] = metrics[f'{task}_accuracy'] = 0.0
            metrics[f'{task}_precision'] = metrics[f'{task}_recall'] = 0.0
            continue

        y_true = np.concatenate(all_labels[task])
        y_prob = np.concatenate(all_logits[task])
        y_pred = (y_prob >= THRESHOLD).astype(float)

        try:
            metrics[task] = float(roc_auc_score(y_true, y_prob))
        except ValueError:
            metrics[task] = 0.0
        
        ap = float(average_precision_score(y_true, y_prob))
        metrics[f'{task}_aupr'] = metrics[f'{task}_mAP'] = ap
        
        p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
        metrics[f'{task}_accuracy']  = float(accuracy_score(y_true, y_pred))
        metrics[f'{task}_precision'] = float(p)
        metrics[f'{task}_recall']    = float(r)
        metrics[f'{task}_f1']        = float(f1)

    # Multilabel helper (Progression, Drug Rec)
    def multilabel_metrics(labels_list, logits_list, prefix):
        if len(labels_list) == 0:
            metrics[prefix] = metrics[f'{prefix}_aupr'] = metrics[f'{prefix}_mAP'] = metrics[f'{prefix}_f1'] = 0.0
            return

        y_true = np.concatenate(labels_list)
        y_prob = np.concatenate(logits_list)
        y_pred = (y_prob >= THRESHOLD).astype(float)

        # Macro AUROC
        aurocs = []
        for i in range(y_true.shape[1]):
            if y_true[:, i].sum() > 0:
                try: aurocs.append(roc_auc_score(y_true[:, i], y_prob[:, i]))
                except ValueError: pass
        metrics[prefix] = float(np.mean(aurocs)) if aurocs else 0.0

        # Macro AUPR (mAP)
        ap = float(average_precision_score(y_true, y_prob, average='macro'))
        metrics[f'{prefix}_aupr'] = metrics[f'{prefix}_mAP'] = ap

        # Micro F1
        _, _, f1, _ = precision_recall_fscore_support(y_true.ravel(), y_pred.ravel(), average='binary', zero_division=0)
        metrics[f'{prefix}_f1'] = float(f1)

    multilabel_metrics(all_labels['progression'], all_logits['progression'], 'progression')
    multilabel_metrics(all_labels['drug_rec'], all_logits['drug_rec'], 'drug_rec')

    # Mean AUROC for early stopping
    available_aurocs = [metrics[t] for t in tasks if len(all_labels[t]) > 0 and t in metrics]
    metrics['mean_auroc'] = float(np.mean(available_aurocs)) if available_aurocs else 0.0
    metrics['loss_dict'] = {k: v / max(n_batches, 1) for k, v in total_loss.items()}

    return metrics


In [12]:
def train(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    start_epoch=1,
    n_epochs=N_EPOCHS,
    patience=PATIENCE,
    device=DEVICE):

    best_mean_auroc   = 0.0
    epochs_no_improve = 0
    history           = []
    recent_checkpoints = [] # Keep last 10 epoch checkpoints in CPU memory for Stochastic Weight Averaging (SWA)
    
    # Mixed Precision Scaler
    scaler = torch.amp.GradScaler('cuda')

    # Initialize/Clear metrics log file only if starting from scratch
    log_mode = 'w' if start_epoch == 1 else 'a'
    with open(os.path.join(CHECKPOINT_DIR, 'metrics.txt'), log_mode) as f:
        if start_epoch == 1:
            f.write(f"=== TRAINING LOG START ===\n")
            f.write(f"Device: {device}\n")
            f.write(f"Batch size: {BATCH_SIZE}, LR: {LR}, Epochs: {n_epochs}\n")
            f.write("="*40 + "\n")
        else:
            f.write(f"\n=== RESUMING FROM EPOCH {start_epoch} ===\n")

    for epoch in range(start_epoch, n_epochs + 1):

        model.train()

        train_loss = {
            t: 0.0
            for t in ['total', 'mortality', 'los_7d', 'readmission', 'progression', 'drug_rec'] 
        }

        n_batches = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{n_epochs}')

        for batch in pbar:

            if batch is None:
                continue

            batch = {
                k: v.to(device) if isinstance(v, torch.Tensor) else v
                for k, v in batch.items()
            }

            optimizer.zero_grad()

            # Autocast for Mixed Precision
            with torch.amp.autocast('cuda'):
                logits = model(batch)
                loss, loss_dict = criterion(logits, batch)

            if torch.isnan(loss):
                print(f"\n[ERROR] NaN Loss detected at Epoch {epoch}, Batch {n_batches}. Skipping.")
                continue

            # Scale loss and backprop
            scaler.scale(loss).backward()
            
            # Unscale for gradient clipping
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

            # Step and update scaler
            scaler.step(optimizer)
            scaler.update()

            for k, v in loss_dict.items():
                train_loss[k] += v

            n_batches += 1

            # Linear LR Warmup
            if epoch <= 5: # Warmup over first 5 epochs
                total_steps = 5 * len(train_loader)
                current_step = (epoch - 1) * len(train_loader) + n_batches
                lr_scale = min(1., float(current_step) / total_steps)
                for pg in optimizer.param_groups:
                    pg['lr'] = LR * lr_scale
            #


            pbar.set_postfix({
                'loss': f"{loss_dict['total']:.3f}",
                'mort': f"{loss_dict['mortality']:.3f}",
                'los':  f"{loss_dict['los_7d']:.3f}",
                'drug': f"{loss_dict['drug_rec']:.3f}",  
            })

        # Average train losses
        train_loss_avg = {
            k: v / max(n_batches, 1)
            for k, v in train_loss.items()
        }

        # Validation
        val_metrics = evaluate(model, val_loader, criterion, device)

        if scheduler is not None:
            scheduler.step(val_metrics['mean_auroc'])

        # Console log & File log
        metrics_msg = (
            f"\nEpoch {epoch:>3} | "
            f"Train loss: {train_loss_avg['total']:.4f} | "
            f"Val mean AUROC: {val_metrics['mean_auroc']:.4f}\n"

            f"\n[MORTALITY]\n"
            f"AUROC: {val_metrics['mortality']:.4f} | "
            f"AUPR: {val_metrics['mortality_aupr']:.4f} | "
            f"F1: {val_metrics['mortality_f1']:.4f}\n"

            f"\n[LOS > 7 DAYS]\n"
            f"AUROC: {val_metrics['los_7d']:.4f} | "
            f"AUPR: {val_metrics['los_7d_aupr']:.4f} | "
            f"F1: {val_metrics['los_7d_f1']:.4f}\n"

            f"\n[READMISSION]\n"
            f"AUROC: {val_metrics['readmission']:.4f} | "
            f"AUPR: {val_metrics['readmission_aupr']:.4f} | "
            f"F1: {val_metrics['readmission_f1']:.4f}\n"

            f"\n[PROGRESSION]\n"
            f"AUROC: {val_metrics['progression']:.4f} | "
            f"AUPR: {val_metrics['progression_aupr']:.4f} | "
            f"F1: {val_metrics['progression_f1']:.4f}\n"

            f"\n[DRUG RECOMMENDATION]\n"
            f"AUROC: {val_metrics['drug_rec']:.4f} | "
            f"AUPR: {val_metrics['drug_rec_aupr']:.4f} | "
            f"F1: {val_metrics['drug_rec_f1']:.4f}\n"

            f"\n[VALIDATION LOSSES]\n"
            f"mort: {val_metrics['loss_dict']['mortality']:.3f} | "
            f"los: {val_metrics['loss_dict']['los_7d']:.3f} | "
            f"readm: {val_metrics['loss_dict']['readmission']:.3f} | "
            f"prog: {val_metrics['loss_dict']['progression']:.3f} | "
            f"drug: {val_metrics['loss_dict']['drug_rec']:.3f}\n"
            f"{'='*40}\n"
        )
        print(metrics_msg)
        with open(os.path.join(CHECKPOINT_DIR, 'metrics.txt'), 'a') as f:
            f.write(metrics_msg)

        # Cleanup
        gc.collect()
        torch.cuda.empty_cache()

        # Save history
        history.append({
            'epoch':      epoch,
            'train_loss': train_loss_avg,
            'val_metrics': val_metrics,
        })

        # Save epoch checkpoint directly to /kaggle/working
        os.makedirs('/kaggle/working', exist_ok=True)
        epoch_path = f"/kaggle/working/epoch_{epoch}.pt"
        raw_model = model.module if hasattr(model, 'module') else model
        torch.save(raw_model.state_dict(), epoch_path)
        print(f"✓ Saved Epoch {epoch} state_dict to {epoch_path}")

        # In-memory SWA: save model state dict to CPU
        recent_checkpoints.append({k: v.cpu().clone() for k, v in model.state_dict().items()})
        if len(recent_checkpoints) > 10:
            recent_checkpoints.pop(0)

        # Save best model
        if val_metrics['mean_auroc'] > best_mean_auroc:

            best_mean_auroc   = val_metrics['mean_auroc']
            epochs_no_improve = 0

            torch.save(
                model.state_dict(),
                os.path.join(CHECKPOINT_DIR, 'best_model.pt')
            )

            print(f"✓ New best mean AUROC: {best_mean_auroc:.4f}")

        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve}/{patience} epochs")

        # Early stopping
        if epochs_no_improve >= patience:
            print(
                f"\nEarly stopping at epoch {epoch}. "
                f"Best mean AUROC: {best_mean_auroc:.4f}"
            )
            break

    # Run Stochastic Weight Averaging (SWA)
    if recent_checkpoints:
        print(f"\n=== Running Stochastic Weight Averaging (SWA) over the last {len(recent_checkpoints)} epochs ===")
        swa_state_dict = {}
        for key in recent_checkpoints[0].keys():
            if recent_checkpoints[0][key].dtype.is_floating_point:
                swa_state_dict[key] = torch.stack([ckpt[key] for ckpt in recent_checkpoints]).mean(dim=0)
            else:
                swa_state_dict[key] = recent_checkpoints[-1][key]
        
        # Save SWA model weights
        torch.save(swa_state_dict, os.path.join(CHECKPOINT_DIR, 'swa_model.pt'))
        print(f"✓ SWA model saved to {CHECKPOINT_DIR}/swa_model.pt")

    return history


In [16]:
MAX_LEN = 644 # Bounded at 99th percentile
import math

# print('Loading data...')
# train_df = pd.read_csv(TRAIN_DF_PATH, dtype={'id': str, 'patient_id': str})
# val_df   = pd.read_csv(VAL_DF_PATH, dtype={'id': str, 'patient_id': str})

# # Add los_7d if not already there
# if 'los_7d' not in train_df.columns:
#     train_df['los_7d'] = (train_df['length_of_stay'] >= 7).astype(float)
#     val_df['los_7d']   = (val_df['length_of_stay']   >= 7).astype(float)

# with open(ADMISSION_NODES_PATH) as f:
#     admission_nodes = json_lib.load(f)
# with open(DIAG_VOCAB_PATH) as f:
#     diag_to_idx = json_lib.load(f)
# with open(DRUG_VOCAB_PATH) as f:
#     drug_to_idx = json_lib.load(f)

# try:
#     patient_cache   = torch.load(PATIENT_CACHE_PATH, map_location='cpu', mmap=True)
#     admission_cache = torch.load(ADMISSION_CACHE_PATH, map_location='cpu', mmap=True)
# except Exception:
#     patient_cache   = torch.load(PATIENT_CACHE_PATH, map_location='cpu')
#     admission_cache = torch.load(ADMISSION_CACHE_PATH, map_location='cpu')

# ──── MEMORY OPTIMIZATION: Convert caches to contiguous shared tensors ────
print('Optimizing Cache Objects into contiguous shared memory Tensors...')
patient_ids = list(patient_cache.keys())
patient_cache_tensor = torch.stack([patient_cache[k] for k in patient_ids]).share_memory_()
patient_to_idx = {str(k): i for i, k in enumerate(patient_ids)}

admission_ids = list(admission_cache.keys())
admission_cache_tensor = torch.stack([admission_cache[k] for k in admission_ids]).share_memory_()
admission_to_idx = {str(k): i for i, k in enumerate(admission_ids)}

# Free massive python dict memory overhead
del patient_cache
del admission_cache
import gc; gc.collect()
# ─────────────────────────────────────────────────────────────────────────

print('Building datasets...')
train_dataset = EHRDataset(
    admissions_df   = train_df,
    timeline_dir    = TIMELINE_DIR,
    admission_nodes = admission_nodes,
    diag_to_idx     = diag_to_idx,
    drug_to_idx     = drug_to_idx,
    patient_cache_tensor   = patient_cache_tensor,
    patient_to_idx         = patient_to_idx,
    admission_cache_tensor = admission_cache_tensor,
    admission_to_idx       = admission_to_idx,
    max_len         = MAX_LEN,
    # ablation_mode   = ablation_mode,
)
val_dataset = EHRDataset(
    admissions_df   = val_df,
    timeline_dir    = TIMELINE_DIR,
    admission_nodes = admission_nodes,
    diag_to_idx     = diag_to_idx,
    drug_to_idx     = drug_to_idx,
    patient_cache_tensor   = patient_cache_tensor,
    patient_to_idx         = patient_to_idx,
    admission_cache_tensor = admission_cache_tensor,
    admission_to_idx       = admission_to_idx,
    max_len         = MAX_LEN,
    # ablation_mode   = ablation_mode,
)

# ──── RAM LEAK REMEDY: persistent_workers=False & num_workers=2 ────
# This completely flushes worker RAM at the end of every epoch.
# Using 2 workers halves background RAM footprint while maintaining parallel speed.
NUM_WORKERS = 2 

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=ehr_collate_fn, num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == 'cuda'),
    persistent_workers=False # Refreshes worker memory every epoch
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=ehr_collate_fn, num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == 'cuda'),
    persistent_workers=False # Refreshes worker memory every epoch
)

# Pos weights
prog_weights = torch.tensor(
    np.load(PROG_WEIGHTS_PATH), dtype=torch.float32
).to(DEVICE)
drug_weights = torch.tensor(
    np.load(DRUG_WEIGHTS_PATH), dtype=torch.float32
).to(DEVICE)

# Compute mortality and readmission pos_weight from train_df
n_train     = len(train_df)
pw_mort     = (train_df['inhospital_dead'] == 0).sum() / (train_df['inhospital_dead'] == 1).sum()
pw_los      = (train_df['los_7d'] == 0).sum() / (train_df['los_7d'] == 1).sum()
pw_readm    = (train_df['readmission_30d'] == 0).sum() / (train_df['readmission_30d'] == 1).sum()

print(f'pos_weight — mortality: {pw_mort:.2f}, los: {pw_los:.2f}, readmission: {pw_readm:.2f}')


# Model

# model = EHRTransformer().to(DEVICE)
model = EHRTransformerBase().to(DEVICE)
# model = EHRModel().to(DEVICE)

if torch.cuda.device_count() > 1:
    print(f"🚀 Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

model.use_gradient_checkpointing = False
print(f'Gradient checkpointing: {model.use_gradient_checkpointing}')


Building datasets...
pos_weight — mortality: 63.93, los: 4.39, readmission: 2.88
Model parameters: 2,484,478
Gradient checkpointing: False


In [ ]:
import time

criterion = EHRLoss(
    pos_weight_mortality   = torch.tensor([pw_mort],  dtype=torch.float32).to(DEVICE),
    pos_weight_los         = torch.tensor([pw_los],   dtype=torch.float32).to(DEVICE),
    pos_weight_readmission = torch.tensor([pw_readm], dtype=torch.float32).to(DEVICE),
    pos_weight_progression = prog_weights,
    pos_weight_drug_rec    = drug_weights,
    w_mortality            = 1.0,
    w_los                  = 1.0,
    w_readmission          = 1.0,
    w_progression          = 1.0,
    w_drug_rec             = 1.0,
    use_focal_loss_mortality = True
).to(DEVICE)

alpha_params = [p for n, p in model.named_parameters() if 'alpha' in n]
base_params  = [p for n, p in model.named_parameters() if 'alpha' not in n]

optimizer = torch.optim.AdamW([
    {'params': base_params,  'lr': LR},
    {'params': alpha_params, 'lr': LR * 10} # Fast convergence for learnable alphas
], weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3
)

# Train
start_time = time.time()
history = train(model, train_loader, val_loader, criterion, optimizer, scheduler)
end_time = time.time()

run_model_time = end_time - start_time
print(f'\nTraining complete in {run_model_time/60:.2f} minutes.')
print(f'Best model saved to {CHECKPOINT_DIR}/best_model.pt')

# Log run time
with open(os.path.join(CHECKPOINT_DIR, 'metrics.txt'), 'a') as f:
    f.write(f"\nTotal Run Time: {run_model_time/60:.2f} minutes\n")

# Final Evaluation on Test Set
print('\n' + '='*40)
print('RUNNING FINAL TEST EVALUATION')
print('='*40)

# Load test data
test_df = pd.read_csv(TEST_DF_PATH, dtype={'id': str, 'patient_id': str})
if 'los_7d' not in test_df.columns:
    test_df['los_7d'] = (test_df['length_of_stay'] >= 7).astype(float)
    
test_dataset = EHRDataset(
    admissions_df   = test_df,
    timeline_dir    = TIMELINE_DIR,
    admission_nodes = admission_nodes,
    diag_to_idx     = diag_to_idx,
    drug_to_idx     = drug_to_idx,
    patient_cache_tensor   = patient_cache_tensor,
    patient_to_idx         = patient_to_idx,
    admission_cache_tensor = admission_cache_tensor,
    admission_to_idx       = admission_to_idx,
    max_len         = MAX_LEN,
    # ablation_mode   = ablation_mode,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=ehr_collate_fn, num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == 'cuda')
)

# 1. Evaluate Best Validation Checkpoint
best_model_path = os.path.join(CHECKPOINT_DIR, 'best_model.pt')
best_metrics = None
if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
    print(f"Loaded best validation model weights from {best_model_path}")
    best_metrics = evaluate(model, test_loader, criterion, DEVICE)

# 2. Evaluate SWA Checkpoint
swa_model_path = os.path.join(CHECKPOINT_DIR, 'swa_model.pt')
swa_metrics = None
if os.path.exists(swa_model_path):
    model.load_state_dict(torch.load(swa_model_path, map_location=DEVICE))
    print(f"Loaded SWA model weights from {swa_model_path}")
    swa_metrics = evaluate(model, test_loader, criterion, DEVICE)

# 3. Print Comparison Table
print('\n' + '='*75)
print('FINAL TEST SET COMPARISON: BEST MODEL vs SWA MODEL')
print('='*75)
print(f'{"Task / Metric":<25} | {"Best Model":<15} | {"SWA Model":<15} | {"Difference":<10}')
print('-'*75)

tasks_to_compare = [
    ('Mortality AUROC', 'mortality'),
    ('Mortality AUPR', 'mortality_aupr'),
    ('LOS > 7d AUROC', 'los_7d'),
    ('LOS > 7d AUPR', 'los_7d_aupr'),
    ('Readmission AUROC', 'readmission'),
    ('Readmission AUPR', 'readmission_aupr'),
    ('Progression AUROC', 'progression'),
    ('Progression AUPR', 'progression_aupr'),
    ('Drug Rec AUROC', 'drug_rec'),
    ('Drug Rec AUPR', 'drug_rec_aupr'),
    ('Mean AUROC (Avg)', 'mean_auroc'),
]

comp_msg = ""
for label, key in tasks_to_compare:
    v_best = best_metrics[key] if (best_metrics and key in best_metrics) else 0.0
    v_swa = swa_metrics[key] if (swa_metrics and key in swa_metrics) else 0.0
    diff = v_swa - v_best
    sign = "+" if diff >= 0 else ""
    print(f'{label:<25} | {v_best:.4f}         | {v_swa:.4f}         | {sign}{diff:.4f}')
    comp_msg += f'{label:<25} | {v_best:.4f}         | {v_swa:.4f}         | {sign}{diff:.4f}\n'
print('='*75)

# Log results to file
with open(os.path.join(CHECKPOINT_DIR, 'metrics.txt'), 'a') as f:
    f.write("\n" + "="*20 + " FINAL TEST RESULTS COMPARISON " + "="*20 + "\n")
    f.write(comp_msg)
    f.write("="*60 + "\n")
    if best_metrics:
        f.write("\nBEST MODEL METRICS:\n" + json.dumps(best_metrics, indent=2) + "\n")
    if swa_metrics:
        f.write("\nSWA MODEL METRICS:\n" + json.dumps(swa_metrics, indent=2) + "\n")

with open(os.path.join(CHECKPOINT_DIR, 'history.json'), 'w') as f:
    json.dump(history, f, indent=2)
print(f'Training history saved to {CHECKPOINT_DIR}/history.json')

metrics_save_path = Path('/kaggle/working/metrics')
metrics_save_path.mkdir(exist_ok=True, parents=True)

metrics_to_save = {
    'best_model': best_metrics,
    'swa_model': swa_metrics
}

with open(os.path.join(metrics_save_path, 'metrics.txt'), 'w') as f:
    json.dump(metrics_to_save, f, indent=2)

print(f'\nSaved final metrics to: {metrics_save_path}')


Epoch 1/50:   4%|▍         | 216/5267 [00:34<11:56,  7.05it/s, loss=4.659, mort=0.036, los=0.948, drug=1.232]